### **FIMbench - `webcontent_utils`**

**Creates** the web-servable content for the FIMbench database. Runs *after* a raw
flood map has been standardized by `processing_floodmap`: it crawls the
standardized maps in the bucket, gathers all metadata into a **unified catalog
core** (`catalog_core.json` + `FIM_extents.geojson`), and turns the extents into
**vector tiles** for the viewer / web app. The webapp is live at- https://tethys.ciroh.org/apps/fimbench-gui/

### **Install**

A plain `uv pip install fimbench` install everything. However, tiling additionally needs the
[`tippecanoe`](https://github.com/felt/tippecanoe) binary on your `PATH`
(`brew install tippecanoe`) - that is a system tool, not a Python package.

In [ ]:
# Installs everything (mbutil included); tippecanoe must be on PATH separately:
!uv pip install fimbench

### **Shared settings**

Building the catalog **reads** the bucket and uploading tiles **writes** to it, so
those steps need credentials; making tiles locally does not. Leave the AWS keys
`None` to use the device's configured credentials.

In [ ]:
from pathlib import Path

out_dir = Path('./out')                          # workspace for catalog + tiles
source_path = out_dir / 'FIM_extents.geojson'    # extents to tile
catalog_path = out_dir / 'catalog_core.json'     # unified metadata catalog

aws_access_key_id = None     # None -> use the device's configured credentials
aws_secret_access_key = None # None -> use the device's configured credentials
region = None                # e.g. 'us-east-1'

### **1) Build the catalog core**

Every keyword shown; only `None`-defaults are optional.

In [ ]:
from fimbench import FIMCatalogBuilder

builder = FIMCatalogBuilder(
    bucket='sdmlab',                          # S3 bucket to crawl
    prefix='FIM_Database/',                   # key prefix of standardized maps
    profile=None,                             # named AWS profile (optional)
    max_str_len=2000,                         # truncate over-long metadata strings
    out_dir=out_dir,                          # where catalog_core.json / geojson are written
    aws_access_key_id=aws_access_key_id,      # explicit key (optional)
    aws_secret_access_key=aws_secret_access_key,  # explicit secret (optional)
    region=region,                            # AWS region (optional)
)
builder.build_catalog(
    out_core='catalog_core.json',             # output catalog filename
)

### **2 a) Make vector tiles locally**

In [ ]:
from fimbench import CatalogandTileManager

manager = CatalogandTileManager(
    out_dir=out_dir,        # workspace (also where local tiles land)
    s3_bucket=None,         # None -> local only (set a bucket to enable uploads)
    s3_prefix=None,         # key prefix for uploaded tiles
    layer_name='fim_extents',  # tile layer name
    min_zoom=3,             # minimum tile zoom level
    max_zoom=14,            # maximum tile zoom level
    aws_access_key_id=None, # only used when s3_bucket is set
    aws_secret_access_key=None,
    region=None,
)
manager.execute(
    source_path=source_path,   # extents (.geojson or .parquet) to tile
    catalog_path=catalog_path, # catalog to attach as tile attributes
    run_tiling=True,           # build the vector tiles
    upload_tiles=False,        # keep tiles local
    upload_json=False,         # keep catalog local
    cleanup=True,              # remove intermediate files when done
    max_workers=None,          # upload parallelism (None -> auto)
)

### **2 b) Make tiles and upload them (+ catalog) to S3**

Setting `s3_bucket` turns on uploads.

In [ ]:
manager = CatalogandTileManager(
    out_dir=out_dir,
    s3_bucket='sdmlab',                       # set a bucket -> uploads enabled
    s3_prefix='FIM_Database/FIM_Viz',         # where tiles/catalog are uploaded
    layer_name='fim_extents',
    min_zoom=3,
    max_zoom=14,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region=region,
)
manager.execute(
    source_path=source_path,
    catalog_path=catalog_path,
    run_tiling=True,    # build tiles
    upload_tiles=True,  # upload the tiles to S3
    upload_json=True,   # upload the catalog to S3
    cleanup=True,
    max_workers=None,
)